# ENSAMBLAJE

In [24]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingRegressor, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.datasets import load_iris

## EJEMPLO 1: VotingClassifier

Vamos a probar el metodo de ensamblaje clasificador de votación con los datos del conjunto de datos moons.
El conjunto de datos "moons" es un conjunto de datos sintético que se utiliza comúnmente en el aprendizaje automático y en el análisis de datos para la demostración y experimentación con algoritmos de clasificación. 

In [25]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42) #n_samplos indica el numero de muestras, noise indica el ruido, random_state indica la semilla
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42) #dividimos el dataset en entrenamiento y test

Creamos un clasificador por votación con los 3 clasificadores que hayamos seleccionado: LogisticRegression, RandomForestClassifier y SVC.

In [26]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

,estimators,"[('lr', ...), ('rf', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


Vamos a evaluar cada modelo del ensamble por separado en el conjunto de prueba y luego al modelo ensamblado

In [27]:
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test)) #.score me devuelve el precisión de cada uno de los modelos respecto a los datos de test

lr = 0.864
rf = 0.896
svc = 0.896


In [28]:
voting_clf.score(X_test, y_test)

0.912

El clasificador de votación tiene un rendimiento mejor que los clasificadores individuales.

Hacemos una predicción con cada uno de los clasificadores que componen el ensamble.

In [29]:
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]

[array([1]), array([1]), array([0])]

Cuando llamamos al método predict() del clasificador de votación, lleva a cabo hard voting. Por ejemplo, el clasificador de votación predice la clase 1 para la primera instancia del conjunto de prueba, porque dos de tres clasificadores predicen esa clase.

In [30]:
voting_clf.predict(X_test[:1])

array([1])

## EJEMPLO 2: BaggingClassifier

 El siguiente código entrena un ensamble de 500 clasificadores de árboles de decisión: cada uno se entrena en 100 instancias de entrenamiento, aleatoriamente muestreadas del conjunto de entrenamiento con reemplazo (este es un ejemplo de bagging, pero, si queremos utilizar pasting, establecemos bootstrap=False). El parámetro n_jobs indica el número de núcleos de CPU que se deben utilizar para el entrenamiento y las predicciones (1 indica todos los núcleos disponibles).

In [31]:
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)

,estimator,DecisionTreeClassifier()
,n_estimators,500
,max_samples,100
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,-1
,random_state,42
,verbose,0


In [32]:
y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.904

## EJEMPLO 3: RandomForestClassifier

El siguiente código entrena clasificador Random Forest con 500 árboles, cada uno limitado a un máximo de 16 nodos terminales, utilizando todos los núcleos de CPU disponibles:

In [33]:
rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)


,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,16
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
y_pred_rf = rnd_clf.predict(X_test)
accuracy_score(y_test, y_pred_rf)

0.912

## EJEMPLO 4: RandomForestClassifier (importancia relativa de cada característica)

Una cualidad estupenda de los Random Forests es que hacen que sea fácil medir la importancia relativa de cada característica

Vamos un ver un ejemplo con el conjunto de datos Iris. El conjunto de datos Iris contiene 150 muestras de flores de iris divididas en tres especies: Iris setosa, Iris versicolor e Iris virginica. Cada muestra tiene cuatro características numéricas: longitud y anchura del sépalo, y longitud y anchura del pétalo. 

El siguiente código entrena un RandomForestClassifier y genera como resultado la importancia de cada característica. Parece que las características más importantes son la longitud (44 %) y la anchura (42 %) del pétalo, mientras que la longitud y la anchura del sépalo no tienen mucha importancia en comparación (11% y 2 %, respectivamente):

In [35]:
iris = load_iris(as_frame=True)
rnd_clf = RandomForestClassifier(n_estimators=500, random_state=42)
rnd_clf.fit(iris.data, iris.target)
for score, name in zip(rnd_clf.feature_importances_, iris.data.columns):
    print(round(score, 2), name)

0.11 sepal length (cm)
0.02 sepal width (cm)
0.44 petal length (cm)
0.42 petal width (cm)


## EJEMPLO 5: AdaBoostClassifier


Cuando entrenamos un clasificador AdaBoost, el algoritmo entrena primero un clasificador base (por ejemplo, un árbol de decisión) y lo utiliza para hacer predicciones en el conjunto de entrenamiento. Después, el algoritmo incrementa el peso relativo de las instancias de entrenamiento mal clasificadas. A continuación, entrena un segundo clasificador, utilizando los pesos actualizados y vuelve a hacer predicciones en el conjunto de entrenamiento, actualiza los pesos de las instancias, y así sucesivamente

El siguiente código entrena un clasificador AdaBoost basado en 30 árboles de un solo nivel 

In [36]:
ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=30,
    learning_rate=0.5, random_state=42)
ada_clf.fit(X_train, y_train)

,estimator,DecisionTreeC...r(max_depth=1)
,n_estimators,30
,learning_rate,0.5
,algorithm,'deprecated'
,random_state,42
,criterion,'gini'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [37]:
y_pred= ada_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.88

## EJEMPLO 6: GradientBoostingRegressor

Al igual que AdaBoost, gradient boosting funciona añadiendo predictores a un ensamble de manera secuencial, de modo que cada uno de ellos corrige a su predecesor. 

Sin embargo, en vez de ajustar la instancia en cada iteración, como hace AdaBoost, este método intenta ajustar el nuevo predictor a los errores residuales cometidos por el predictor anterior. 

Veamos un sencillo ejemplo de regresión que utiliza árboles de decisión como predictores básicos, para lo cual primero, vamos a generar un conjunto de datos cuadrático con ruido:

In [38]:
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3 * X[:, 0] ** 2 + 0.05 * np.random.randn(100)  # y = 3x² + Gaussian noise

Vamos a crear un modelo Gradient Boosting Regressor que internamente utiliza 3 árboles de decisión con una profundidad máxima de 2 niveles y una tasa de aprendizaje de 1.0,

In [39]:
gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt.fit(X, y)

,loss,'squared_error'
,learning_rate,1.0
,n_estimators,3
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,2
,min_impurity_decrease,0.0
,init,None


In [40]:
print("y min:", y.min())
print("y max:", y.max())
y_pred = gbrt.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print("RMSE:", rmse)

y min: -0.07017795621787389
y max: 0.759622208242618
RMSE: 0.07097928528094154


Para encontrar el número óptimo de árboles, podemos utilizar la validación cruzada usando GridSearchCV o RandomizedSearchCV, como siempre, pero hay una manera más sencilla: si establecemos el hiperparámetro n_iter_no_change como un valor entero, por ejemplo, 10, el GradientBoostingRegressor dejará de forma automática de añadir árboles durante el entrenamiento si ve que los últimos 10 árboles no han ayudado

In [41]:
gbrt_best = GradientBoostingRegressor( max_depth=2, learning_rate=0.05, n_estimators=500, n_iter_no_change=10, random_state=42)
gbrt_best.fit(X, y)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,500
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,2
,min_impurity_decrease,0.0
,init,None


In [42]:
print("y min:", y.min())
print("y max:", y.max())
y_pred = gbrt_best.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print("RMSE:", rmse)

y min: -0.07017795621787389
y max: 0.759622208242618
RMSE: 0.036585111000770935


In [43]:
gbrt_best.n_estimators_

92

## EJEMPLO 7: StackingClassifier

Por ejemplo, podemos sustituir el VotingClassifier que hemos utilizado al principio de este capítulo con el conjunto de datos moons por un StackingClassifier:

In [44]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [45]:
stacking_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    final_estimator=RandomForestClassifier(random_state=43),
    cv=5  # number of cross-validation folds
)
stacking_clf.fit(X_train, y_train)

,estimators,"[('lr', ...), ('rf', ...), ...]"
,final_estimator,RandomForestC...ndom_state=43)
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [46]:
stacking_clf.score(X_test, y_test)

0.928